### 1. Dataset Preprocessing and Format Conversion
The official MOT16 dataset provides data as extracted frames (`img1` directories) to prevent video compression artifacts. This script reads the raw frames and Ground Truth (`gt.txt`) files, filters for the `pedestrian` class (class ID 1) with a visibility threshold to ignore heavily occluded individuals, and normalizes the bounding box coordinates to meet the YOLO training format requirements.

In [3]:
import os
import shutil
import pandas as pd
import configparser
from tqdm import tqdm

MOT_PATH = '/kaggle/input/datasets/takshmandar/mot16-dataset/train'
YOLO_PATH = '/kaggle/working/yolo_dataset'

for split in ['train', 'val']:
    os.makedirs(f'{YOLO_PATH}/images/{split}', exist_ok=True)
    os.makedirs(f'{YOLO_PATH}/labels/{split}', exist_ok=True)

split_seqs = {
    'train': ['MOT16-02', 'MOT16-04', 'MOT16-05', 'MOT16-09', 'MOT16-10'],
    'val': ['MOT16-11', 'MOT16-13']
}

for split, seq_list in split_seqs.items():
    print(f"Processing {split} set...")
    for seq in seq_list:
        seq_path = os.path.join(MOT_PATH, seq)
        
        if not os.path.exists(seq_path):
            print(f"Warning: {seq_path} not found. Please verify the MOT_PATH.")
            continue
            
        config = configparser.ConfigParser()
        config.read(os.path.join(seq_path, 'seqinfo.ini'))
        im_w = float(config['Sequence']['imWidth'])
        im_h = float(config['Sequence']['imHeight'])
        
        gt_path = os.path.join(seq_path, 'gt', 'gt.txt')
        cols = ['frame', 'id', 'bb_left', 'bb_top', 'bb_width', 'bb_height', 'conf', 'class', 'vis']
        gt = pd.read_csv(gt_path, header=None, names=cols)
        
        gt = gt[(gt['class'] == 1) & (gt['vis'] > 0.2)]
        
        frames = gt.groupby('frame')
        img_dir = os.path.join(seq_path, 'img1')
        
        for frame_id, frame_data in tqdm(frames, desc=seq):
            img_name = f"{frame_id:06d}.jpg"
            src_img = os.path.join(img_dir, img_name)
            
            if not os.path.exists(src_img):
                continue
                
            new_name = f"{seq}_{img_name}"
            dst_img = os.path.join(YOLO_PATH, 'images', split, new_name)
            dst_lbl = os.path.join(YOLO_PATH, 'labels', split, new_name.replace('.jpg', '.txt'))
            
            shutil.copy(src_img, dst_img)
            
            with open(dst_lbl, 'w') as f:
                for _, row in frame_data.iterrows():
                    x_center = (row['bb_left'] + row['bb_width'] / 2) / im_w
                    y_center = (row['bb_top'] + row['bb_height'] / 2) / im_h
                    w = row['bb_width'] / im_w
                    h = row['bb_height'] / im_h
                    
                    x_center, y_center = max(0, min(1, x_center)), max(0, min(1, y_center))
                    w, h = max(0, min(1, w)), max(0, min(1, h))
                    
                    f.write(f"0 {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}\n")

print("Dataset conversion completed successfully.")

Processing train set...


MOT16-10: 100%|██████████| 654/654 [00:01<00:00, 401.64it/s]


Processing val set...


MOT16-13: 100%|██████████| 750/750 [00:01<00:00, 426.58it/s]

Dataset conversion completed successfully.


### 2. YOLO Configuration
Generating the `data.yaml` file required by the YOLO training pipeline. This file defines the paths to the dataset splits and maps the class indices.

In [4]:
yaml_content = """
path: /kaggle/working/yolo_dataset
train: images/train
val: images/val

names:
  0: person
"""

with open('/kaggle/working/yolo_dataset/data.yaml', 'w') as f:
    f.write(yaml_content.strip())
    
print("data.yaml created successfully.")

data.yaml created successfully.


### 3. Environment Setup
Installing the `ultralytics` framework which provides the YOLOv8 implementation. The quiet flag is utilized to suppress extensive dependency resolution logs, keeping the notebook clean for presentation.

In [5]:
!pip install -q ultralytics
import ultralytics
ultralytics.checks()

Ultralytics 8.4.102 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Setup complete ✅ (4 CPUs, 31.3 GB RAM, 7035.6/8062.4 GB disk)


### 4. Model Fine-Tuning
We initialize a pre-trained YOLOv8 Nano (`yolov8n.pt`) model. The Nano variant is specifically selected to guarantee high frames-per-second (FPS) and low latency during real-time inference on embedded systems. The model is fine-tuned on our prepared MOT16 pedestrian dataset.

In [4]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data='/kaggle/working/yolo_dataset/data.yaml',
    epochs=15,
    imgsz=640,
    batch=32,
    project='/kaggle/working/runs',
    name='crowd_yolo',
    device='0'
)

Ultralytics 8.4.102 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=crowd_yolo, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto,

### 5. Model Export for Embedded Optimization
Exporting the fine-tuned PyTorch model to the ONNX (Open Neural Network Exchange) format. This conversion strips away PyTorch-specific dependencies and optimizes the computational graph, which is a critical requirement for achieving real-time inference (low latency) on resource-constrained embedded systems.

In [5]:
from ultralytics import YOLO

# Load the best weights from the fine-tuning phase
best_model = YOLO('/kaggle/working/runs/crowd_yolo/weights/best.pt')

# Export to ONNX format with graph simplification
exported_path = best_model.export(format='onnx', simplify=True)

print(f"ONNX Export Successful. File located at: {exported_path}")

Ultralytics 8.4.102 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from '/kaggle/working/runs/crowd_yolo/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (5.9 MB)
requirements: Ultralytics requirements ['onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 251ms
Prepared 2 packages in 289ms
Installed 2 packages in 13ms
 + onnxruntime==1.27.0
 + onnxslim==0.1.94

requirements: AutoUpdate success ✅ 0.9s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.22.0 opset 20...
ONNX: slimming with onnxslim 0.1.94...
ONNX: export success ✅ 2.4s, saved

### 6. Exporting Training Artifacts
Compressing the training artifacts, including the PyTorch weights, the optimized ONNX model, and the training evaluation graphs. This creates a lightweight archive for local deployment and report generation, explicitly excluding the massive image dataset.

In [6]:
import shutil
import os

runs_dir = '/kaggle/working/runs'
output_zip = '/kaggle/working/yolo_training_results'

shutil.make_archive(output_zip, 'zip', runs_dir)

zip_size_mb = os.path.getsize(f"{output_zip}.zip") / (1024 * 1024)
print(f"Archive created successfully: {output_zip}.zip ({zip_size_mb:.2f} MB)")

Archive created successfully: /kaggle/working/yolo_training_results.zip (27.64 MB)


### 7. Advanced Training with YOLOv8 Medium
Upgrading the pipeline to YOLOv8 Medium (`yolov8m.pt`) to leverage deeper feature extraction capabilities. We extend the training duration to 35 epochs and reduce the batch size to 16 to accommodate the larger parameter space, aiming to minimize false positives like window patterns.

In [ ]:
from ultralytics import YOLO

# Load the Medium YOLOv8 model weights
model = YOLO('yolov8m.pt')

results = model.train(
    data='/kaggle/working/yolo_dataset/data.yaml',
    epochs=35,
    imgsz=640,
    batch=16,
    project='/kaggle/working/runs',
    name='crowd_yolo_medium',
    device='0'
)

Ultralytics 8.4.102 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=35, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=crowd_yolo_medium, nbs=64, nms=False, opset=None, optimize=False, optimize

In [7]:
from ultralytics import YOLO

# Load the best weights from the fine-tuning phase
best_model = YOLO('/kaggle/working/runs/crowd_yolo_medium/weights/best.pt')

# Export to ONNX format with graph simplification
exported_path = best_model.export(format='onnx', simplify=True)

print(f"ONNX Export Successful. File located at: {exported_path}")

Ultralytics 8.4.102 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
Model summary (fused): 93 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs

PyTorch: starting from '/kaggle/working/runs/crowd_yolo_medium/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (49.6 MB)
requirements: Ultralytics requirements ['onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 384ms
Prepared 2 packages in 334ms
Installed 2 packages in 13ms
 + onnxruntime==1.27.0
 + onnxslim==0.1.94

requirements: AutoUpdate success ✅ 1.1s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.22.0 opset 20...
ONNX: slimming with onnxslim 0.1.94...
ONNX: export success ✅ 4

### 8. We had overfit with 35 epoch training with YOLOv8 Medium
to prevent overfitting we defined `patience` and `dropout`.

In [9]:
from ultralytics import YOLO

model = YOLO('yolov8m.pt')

results = model.train(
    data='/kaggle/working/yolo_dataset/data.yaml',
    epochs=15,
    patience=5,
    imgsz=640,
    batch=16,
    freeze=10,
    dropout=0.2,
    project='/kaggle/working/runs',
    name='crowd_yolo_medium_robust',
    device='0'
)

Ultralytics 8.4.102 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.2, dynamic=False, embed=None, end2end=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=crowd_yolo_medium_robust, nbs=64, nms=False, opset=None, optimize=False, opt

In [10]:
from ultralytics import YOLO

# Load the best weights from the fine-tuning phase
best_model = YOLO('/kaggle/working/runs/crowd_yolo_medium_robust/weights/best.pt')

# Export to ONNX format with graph simplification
exported_path = best_model.export(format='onnx', simplify=True)

print(f"ONNX Export Successful. File located at: {exported_path}")

Ultralytics 8.4.102 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
Model summary (fused): 93 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs

PyTorch: starting from '/kaggle/working/runs/crowd_yolo_medium_robust/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (49.6 MB)

ONNX: starting export with onnx 1.22.0 opset 20...
ONNX: slimming with onnxslim 0.1.94...
ONNX: export success ✅ 2.8s, saved as '/kaggle/working/runs/crowd_yolo_medium_robust/weights/best.onnx' (98.8 MB)

Export complete (4.2s)
Results saved to /kaggle/working/runs/crowd_yolo_medium_robust/weights/best.onnx
Predict:         yolo predict task=detect model=/kaggle/working/runs/crowd_yolo_medium_robust/weights/best.onnx imgsz=640 
Validate:        yolo val task=detect model=/kaggle/working/runs/crowd_yolo_medium_robust/weights/best.onnx imgsz=640 data=/kaggle/working/yolo_dataset/data.yaml  
Visualize:       https://netron.app
ONNX Export Successful. File

In [11]:
import shutil
import os

runs_dir = '/kaggle/working/runs'
output_zip = '/kaggle/working/yolo_training_results_robust_epoch'

shutil.make_archive(output_zip, 'zip', runs_dir)

zip_size_mb = os.path.getsize(f"{output_zip}.zip") / (1024 * 1024)
print(f"Archive created successfully: {output_zip}.zip ({zip_size_mb:.2f} MB)")

Archive created successfully: /kaggle/working/yolo_training_results_robust_epoch.zip (361.46 MB)
